In [1]:
import os
import pandas as pd

# ==============================================================================
# 1. تحديد مسار المجلد وقائمة الملفات الستة بالترتيب
# ==============================================================================
DATA_DIR = r"d:\vscode\NASA\DATA"

FILES = [
    "01_NASA_POWER_Climate.csv",
    "02_ERA5_Land_Hydrology.csv",
    "03_SMAP_SoilMoisture.csv",
    "04_GPM_Precipitation.csv",
    "05_MODIS_NDVI_EVI_ET.csv",
    "06_GLDAS_Terrain_Soil.csv",
]

print("=" * 80)
print("🚀 بدء عملية الدمج الاحترافي للملفات الستة (2016 - 2025)")
print("=" * 80)

df_master = None
loaded_count = 0

# ==============================================================================
# 2. قراءة كل ملف والتحقق من أبعاده ودمجه مع الحفاظ على البيانات الأصلية
# ==============================================================================
for fname in FILES:
    fpath = os.path.join(DATA_DIR, fname)
    if not os.path.exists(fpath):
        raise FileNotFoundError(f"❌ الملف غير موجود في المسار: {fpath}")

    df_part = pd.read_csv(fpath)

    # توحيد صيغة عمود التاريخ YYYY-MM-DD لضمان تطابق الربط 100%
    df_part["Date"] = pd.to_datetime(df_part["Date"]).dt.strftime("%Y-%m-%d")
    df_part["Governorate"] = df_part["Governorate"].astype(str).str.strip()

    # التأكد من عدم وجود تكرار في مفتاح الربط (Date + Governorate)
    df_part = df_part.drop_duplicates(subset=["Date", "Governorate"]).reset_index(drop=True)

    print(f"✓ تم تحميل: {fname:<32} | الأبعاد: {df_part.shape[0]} صف × {df_part.shape[1]} عمود")

    if df_master is None:
        df_master = df_part
    else:
        # الحفاظ على أسماء الأعمدة الأصلية ومنع تكرار أي عمود موجود مسبقاً
        new_cols = ["Date", "Governorate"] + [
            c for c in df_part.columns if c not in df_master.columns
        ]
        df_master = pd.merge(
            df_master,
            df_part[new_cols],
            on=["Date", "Governorate"],
            how="outer"
        )
    loaded_count += 1

# ==============================================================================
# 3. الترتيب الاحترافي حسب التاريخ (Date) ثم المحافظة (Governorate)
# ==============================================================================
df_master = df_master.sort_values(by=["Date", "Governorate"], ascending=[True, True]).reset_index(drop=True)

# وضع عمودي Date و Governorate في مقدمة الجدول لسهولة القراءة والتحليل
ordered_cols = ["Date", "Governorate"] + [
    c for c in df_master.columns if c not in ["Date", "Governorate"]
]
df_master = df_master[ordered_cols]

# ==============================================================================
# 4. حفظ الملف النهائي الموحد وطباعة ملخص الجودة
# ==============================================================================
output_file = os.path.join(DATA_DIR, "Egypt_NASA_10Years_Master_Harmonized.csv")
df_master.to_csv(output_file, index=False, encoding="utf-8-sig")

print("\n" + "=" * 80)
print(f"🎉 اكتمل دمج الملفات الـ {loaded_count} بنجاح تام!")
print(f"📁 مسار الملف النهائي: {output_file}")
print(f"📐 الأبعاد النهائية: {df_master.shape[0]} صف × {df_master.shape[1]} عمود")
print(f"📅 النطاق الزمني: من {df_master['Date'].min()} إلى {df_master['Date'].max()}")
print(f"🌍 عدد المحافظات: {df_master['Governorate'].nunique()} محافظة")
print(f"🧹 إجمالي القيم المفقودة (Nulls): {df_master.isnull().sum().sum()}")
print("=" * 80)
print("\n📋 قائمة بجميع الأعمدة المحفوظة في الملف النهائي:")
for idx, col in enumerate(df_master.columns, start=1):
    print(f"   {idx:02d}. {col}")

🚀 بدء عملية الدمج الاحترافي للملفات الستة (2016 - 2025)
✓ تم تحميل: 01_NASA_POWER_Climate.csv        | الأبعاد: 5110 صف × 12 عمود
✓ تم تحميل: 02_ERA5_Land_Hydrology.csv       | الأبعاد: 5110 صف × 8 عمود
✓ تم تحميل: 03_SMAP_SoilMoisture.csv         | الأبعاد: 5110 صف × 6 عمود
✓ تم تحميل: 04_GPM_Precipitation.csv         | الأبعاد: 5110 صف × 4 عمود
✓ تم تحميل: 05_MODIS_NDVI_EVI_ET.csv         | الأبعاد: 5110 صف × 6 عمود
✓ تم تحميل: 06_GLDAS_Terrain_Soil.csv        | الأبعاد: 5110 صف × 14 عمود

🎉 اكتمل دمج الملفات الـ 6 بنجاح تام!
📁 مسار الملف النهائي: d:\vscode\NASA\DATA\Egypt_NASA_10Years_Master_Harmonized.csv
📐 الأبعاد النهائية: 5110 صف × 40 عمود
📅 النطاق الزمني: من 2016-01-01 إلى 2025-12-19
🌍 عدد المحافظات: 14 محافظة
🧹 إجمالي القيم المفقودة (Nulls): 730

📋 قائمة بجميع الأعمدة المحفوظة في الملف النهائي:
   01. Date
   02. Governorate
   03. POWER_Temp_Mean_C
   04. POWER_Temp_Min_C
   05. POWER_Temp_Max_C
   06. POWER_Humidity_Pct
   07. POWER_Solar_Rad_MJ_m2
   08. POWER_Precip_mm_day